# 🏥 Medical Imaging using OCT Images — Retinal Disease Classification

This project classifies retinal diseases from OCT (Optical Coherence Tomography) images using a Convolutional Neural Network (CNN).

**Classes:**
- CNV (Choroidal Neovascularization)
- DME (Diabetic Macular Edema)
- DRUSEN
- NORMAL

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('Libraries imported successfully!')

## Step 2: Define Parameters

In [ ]:
# Image parameters
IMG_SIZE = 128          # Resize all images to 128x128
BATCH_SIZE = 32
EPOCHS = 20
NUM_CLASSES = 4         # CNV, DME, DRUSEN, NORMAL
LEARNING_RATE = 0.001

# Class labels
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

print('Parameters defined!')
print(f'Image Size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Classes: {CLASS_NAMES}')

## Step 3: Load and Preprocess Data

> **Note:** Dataset should be organized as:
> ```
> dataset/
>     train/
>         CNV/
>         DME/
>         DRUSEN/
>         NORMAL/
>     test/
>         CNV/
>         DME/
>         DRUSEN/
>         NORMAL/
> ```
> Download dataset from: https://www.kaggle.com/datasets/paultimothymooney/kermany2018

In [ ]:
# Define dataset paths
TRAIN_DIR = 'dataset/train'
TEST_DIR  = 'dataset/test'

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2     # 20% for validation
)

# Only rescaling for test data
test_datagen = ImageDataGenerator(rescale=1./255)

# Training data generator
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    color_mode='grayscale',
    shuffle=True
)

# Validation data generator
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    color_mode='grayscale',
    shuffle=False
)

# Test data generator
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

print('\nClass Indices:', train_generator.class_indices)

## Step 4: Visualize Sample Images

In [ ]:
# Plot sample images from each class
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Sample OCT Images — Retinal Disease Classes', fontsize=14, fontweight='bold')

for i, class_name in enumerate(CLASS_NAMES):
    class_path = os.path.join(TRAIN_DIR, class_name)
    img_file = os.listdir(class_path)[0]
    img = cv2.imread(os.path.join(class_path, img_file), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(class_name, fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Step 5: Build the CNN Model

In [ ]:
def build_cnn_model(input_shape, num_classes):
    model = models.Sequential([

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Fully Connected Layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        # Output Layer
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Build model
input_shape = (IMG_SIZE, IMG_SIZE, 1)  # Grayscale images
model = build_cnn_model(input_shape, NUM_CLASSES)

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()

## Step 6: Train the Model

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_oct_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
print('Training started...')
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)
print('Training complete!')

## Step 7: Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', color='blue')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='orange')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Step 8: Evaluate the Model

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f'\nTest Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_accuracy*100:.2f}%')

## Step 9: Classification Report & Confusion Matrix

In [ ]:
# Predictions
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Classification Report
print('\nClassification Report:')
print('=' * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — OCT Retinal Disease Classification', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## Step 10: Predict on a Single Image

In [ ]:
def predict_single_image(img_path, model, class_names, img_size=128):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, (img_size, img_size))
    img_normalized = img_resized / 255.0
    img_input = img_normalized.reshape(1, img_size, img_size, 1)

    prediction = model.predict(img_input)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    plt.figure(figsize=(5, 5))
    plt.imshow(img_resized, cmap='gray')
    plt.title(f'Predicted: {predicted_class}\nConfidence: {confidence:.2f}%',
              fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.show()

    print(f'Predicted Class : {predicted_class}')
    print(f'Confidence      : {confidence:.2f}%')
    return predicted_class, confidence

# Example usage — replace with your image path
# predict_single_image('path/to/your/oct_image.jpeg', model, CLASS_NAMES)

## Step 11: Save the Model

In [ ]:
# Save the final model
model.save('oct_retinal_disease_cnn_model.h5')
print('Model saved as oct_retinal_disease_cnn_model.h5')